## Documento de Requerimientos del Sistema (DRS) - Fase 3: Arquitectura y API Robusta

Avanzamos a la **Fase 3**, centrada en la optimización de consultas mediante sistemas de almacenamiento en caché (Redis), cumpliendo exactamente con el Tema 3 de la planificación.

El objetivo de esta fase es enseñar a los estudiantes cómo evitar que la base de datos colapse ante consultas repetitivas (como listar el catálogo o reconstruir el árbol de referidos), aplicando el patrón *Cache-Aside*.

### Instrucciones de Ejecución para la IA (Fase 3)

Copia y pega este texto en tu entorno (Windsurf) para generar la integración con Redis:

```markdown
> **Objetivo:** Integrar Redis como sistema de almacenamiento en caché para optimizar las consultas de lectura recurrentes (Catálogo y Red Multinivel), reduciendo la latencia y la carga sobre PostgreSQL.
> 
> **Paso 1: Infraestructura (Docker)**
> Modifica el archivo `docker-compose.yml` existente agregando un nuevo servicio para Redis (utiliza la imagen `redis:alpine`). Expón el puerto 6379 y asegúrate de que el backend en Go dependa tanto de PostgreSQL como de Redis.
> 
> **Paso 2: Cliente de Redis (`config/redis.go`)**
> Instala la librería oficial de Redis para Go (`github.com/redis/go-redis/v9`). Crea el archivo `config/redis.go` e inicializa el cliente de conexión (`redis.NewClient`). Asegúrate de leer la URI o el host desde las variables de entorno.
> 
> **Paso 3: Caché en el Catálogo de Productos**
> Modifica el controlador `GetProductos` (o la función correspondiente en el repositorio) para implementar el patrón Cache-Aside:
> 1. Intenta leer la clave `productos:all` desde Redis.
> 2. Si existe (Cache Hit), deserializa el JSON y retorna la respuesta inmediatamente (sin tocar PostgreSQL).
> 3. Si no existe (Cache Miss), consulta PostgreSQL, serializa el resultado a JSON, guárdalo en Redis con un tiempo de expiración (TTL) de 5 minutos, y retorna la respuesta.
> 
> **Paso 4: Caché en la Red Multinivel (Operación Costosa)**
> La reconstrucción del árbol de referidos es pesada. Modifica `GetRed` para que busque en Redis la clave `red:tree:{email_usuario}`. Si hay un *Cache Miss*, arma el árbol desde PostgreSQL, guárdalo en Redis con un TTL de 10 minutos y retorna.

```

### ¿Cómo visualizar el avance en pantalla?

Esta es una de las fases más visuales para explicar el rendimiento a los estudiantes usando las herramientas de desarrollador:

1. Levanta el proyecto con `docker-compose up -d`.
2. Entra a la aplicación React y navega al **Dashboard** (ruta `/` para el administrador). Al cargar el árbol de "Mi Red", abre la pestaña **Network** (Red) del navegador (o usa Postman). Observarás que la primera petición a `/api/red` tarda, por ejemplo, **120ms** (porque va a PostgreSQL y arma el árbol).
3. Presiona F5 para recargar la página. Al mirar nuevamente la pestaña Network, el tiempo de respuesta de `/api/red` habrá caído drásticamente (ej. **3ms a 8ms**), demostrando que la petición ni siquiera tocó la base de datos, sino que fuotor de transferencias y *checkout*.